# Lesson 1 — Train your first neural network from scratch

**The one big idea:** *training* means repeatedly nudging a model's parameters to make its errors smaller. Every flavor of ML you've heard of — fine-tuning, LoRA, RLHF — is this same loop at different scales, usually starting from someone else's already-trained parameters.

By the end of this notebook you will have written, by hand, the five lines that constitute ~90% of all machine learning.

Run each cell top to bottom (`Shift+Enter`). Everything here runs in seconds, mostly on CPU.

In [ ]:
import json
from pathlib import Path
import torch

def load_config():
    for p in [Path("workshop_config.json"), Path("../../workshop_config.json"), Path("../../../workshop_config.json")]:
        if p.exists():
            return json.loads(p.read_text())
    if torch.cuda.is_available():
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        return {"device": "cuda", "dtype": "bfloat16", "vram_gb": vram,
                "tier": "gpu-high" if vram >= 12 else "gpu-low"}
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return {"device": "mps", "dtype": "float16", "vram_gb": None, "tier": "gpu-low"}
    return {"device": "cpu", "dtype": "float32", "vram_gb": None, "tier": "cpu"}

CFG = load_config()
DEVICE, DTYPE, TIER = CFG["device"], getattr(torch, CFG["dtype"]), CFG["tier"]
print(f"Workshop tier: {TIER} | device: {DEVICE} | dtype: {DTYPE}")

In [ ]:
import torch
import matplotlib.pyplot as plt

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Tensors — the raw material

A tensor is just an n-dimensional array (like NumPy's ndarray) with two superpowers:
1. It can live on your **GPU**, where matrix math is ~100x faster.
2. It can **remember the operations performed on it** (that's the next section).

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.eye(2) * 10

print("shapes:", tuple(a.shape), tuple(b.shape))
print("matrix product:\n", a @ b)
print("elementwise:\n", a * b)

x_cpu = torch.randn(1000, 1000)
if torch.cuda.is_available():
    x_gpu = x_cpu.to("cuda")
    print("same values, different device:", x_cpu.device, "->", x_gpu.device)

## 2. Autograd — the magic trick

PyTorch secretly records every operation performed on a tensor flagged `requires_grad=True`. Calling `loss.backward()` then replays that recording **backwards**, computing the derivative of the loss with respect to every parameter — this is *backpropagation*, and it's why nobody computes gradients by hand anymore.

In [ ]:
# One parameter w. We want the value of w that makes (w*x - y)^2 smallest.
w = torch.tensor(3.0, requires_grad=True)
x = torch.tensor(2.0)
y = torch.tensor(10.0)

loss = (w * x - y) ** 2        # loss = (2w - 10)^2, minimum at w = 5
loss.backward()                # fills w.grad with d(loss)/dw

print("loss:", loss.item())
print("autograd says d(loss)/dw =", w.grad.item())
print("by hand: 2*x*(w*x - y)  =", (2 * x * (w * x - y)).item())   # calculus agrees

In [ ]:
# Gradient descent by hand: step against the gradient, watch w walk toward 5.
w = torch.tensor(0.0, requires_grad=True)
lr = 0.1                                   # learning rate = step size
for step in range(20):
    loss = (w * x - y) ** 2
    loss.backward()
    with torch.no_grad():                   # don't record this nudge
        w -= lr * w.grad                    # downhill!
    w.grad = None                           # clear for next step
    if step % 4 == 0:
        print(f"step {step:2d}   w = {w.item():.4f}   loss = {loss.item():.4f}")
print(f"final w = {w.item():.4f}   (true answer: 5)")

## 3. A real (tiny) neural network

`nn.Linear` is a weight matrix + bias — a trainable linear function. `ReLU` is a nonlinear filter (negative values → 0). Stack linear → ReLU → linear → ReLU → linear and you get a network that can approximate almost any function. The `parameters` are exactly the `w`s from above, just ~4,000 of them instead of 1.

In [ ]:
import torch.nn as nn

class TinyMLP(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
model = TinyMLP()
print(f"parameter count: {sum(p.numel() for p in model.parameters())}")
print(model)

## 4. THE training loop

Memorize these five lines. `optimizer` is just the `w -= lr * w.grad` nudge from §2, automated for all parameters at once.

In [ ]:
# Data: 200 noisy samples of y = sin(x)
N = 200
x_train = torch.rand(N, 1) * 6 - 3
y_train = torch.sin(x_train) + 0.05 * torch.randn(N, 1)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

with torch.no_grad():                       # snapshot before training, for comparison
    x_line = torch.linspace(-3, 3, 300).unsqueeze(1)
    pred_before = model(x_line)

def train(model, x, y, epochs=2000):
    for epoch in range(epochs):
        y_pred = model(x)               # 1. forward: make a guess
        loss = loss_fn(y_pred, y)       # 2. measure how wrong it is
        optimizer.zero_grad()           # 3. clear last step's gradients
        loss.backward()                 # 4. backprop: fill every .grad
        optimizer.step()                # 5. nudge every parameter downhill
        if epoch % 500 == 0:
            print(f"epoch {epoch:4d}   loss {loss.item():.5f}")

train(model, x_train, y_train)

In [ ]:
with torch.no_grad():
    pred_after = model(x_line)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, pred, title in [(axes[0], pred_before, "before training"),
                        (axes[1], pred_after, "after 2000 steps")]:
    ax.scatter(x_train, y_train, s=6, alpha=0.4, label="data")
    ax.plot(x_line, pred, color="red", lw=2, label="model")
    ax.set_title(title)
    ax.legend()
plt.show()

## 5. What just happened — and why it matters

The red line started as random garbage and became a sine wave, purely from step 5 of the loop nudging 4,353 parameters downhill. **Nobody told the model what a sine was.**

Now the reframe that unlocks everything else:

- Instead of 4,353 random parameters, start from **7 billion parameters someone already trained** on most of the internet.
- Instead of `(x, sin(x))` pairs, use `(prompt, good_answer)` pairs.
- Instead of updating all parameters, freeze 99.9% and nudge a small overlay (**LoRA**).
- Run steps 1–5 on the GPU (`.to("cuda")`).

That's fine-tuning. That's Stage 3 of the roadmap. The loop does not change.

In [ ]:
# EXERCISES — edit cells above and re-run. Each one teaches something specific.
#
# 1. Harder target: in the data cell, replace y_train with
#       y_train = torch.sin(2*x_train) + 0.5*torch.cos(3*x_train) + 0.05*torch.randn(N, 1)
#    Create a fresh model (re-run that cell too). Does it still fit?
#    Try hidden=128, or epochs=5000. Watch the loss.
#
# 2. Learning rate: set lr=1.0 in the Adam line with a fresh model.
#    What happens to the loss? (This is 'divergence' — step size matters.)
#    Now try lr=1e-5. What happens? (Too small = too slow.)
#
# 3. Capacity: compare parameter counts for hidden=16 vs 64 vs 256.
#    Bigger always fits better... or does it? Try hidden=256 on only N=20 data points.
#    (Sneak preview of 'overfitting' — Stage 2's main topic.)
#
# 4. GPU: add model.to('cuda'), x_train.to('cuda'), y_train.to('cuda') and
#    put %%time at the top of the training cell. The GPU may well LOSE on a problem
#    this tiny — per-step overhead dominates. That flips as models grow.

print("Your turn — edit, re-run, observe. Write down what surprised you.")

## Next: Lesson 2 — Tokenization and a tiny transformer

Numbers in, numbers out worked for `sin(x)`. But language models eat *text*. Lesson 2: how text becomes tokens, and we train a mini character-level language model from absolute scratch — it will hallucinate Shakespeare.